# SparkSession, SparkContext, SparkConf, and the Complete Spark Application Lifecycle


## Learning objectives

- `SparkSession`, `SparkContext`, and `SparkConf`
- The relationship between these objects
- Static and runtime Spark configurations
- Configuration precedence
- Driver initialization
- Cluster-manager communication
- Executor startup and registration
- Job, stage, and task creation
- Fault handling
- Application shutdown
- Spark UI and History Server
- Common notebook and production mistakes
- Interview questions and scenarios

# 1. Big Picture

```text
User Code
   │
   ▼
SparkSession
   │
   ├── DataFrame API
   ├── Spark SQL
   ├── Catalog
   └── Runtime SQL Configuration
   │
   ▼
SparkContext
   │
   ├── Cluster Connection
   ├── DAGScheduler
   ├── TaskScheduler
   ├── RDD API
   ├── Broadcast Variables
   └── Accumulators
   │
   ▼
SparkConf
   ├── Application Name
   ├── Master URL
   ├── Driver Settings
   ├── Executor Settings
   └── Other Spark Properties
```

In simple terms:

- **SparkConf** stores configuration.
- **SparkContext** connects the application to Spark execution.
- **SparkSession** is the modern high-level entry point.

In [ ]:
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, sum as spark_sum

print("PySpark classes imported successfully.")

# 2. SparkConf

`SparkConf` stores Spark application configuration.

Common properties include:

- `spark.app.name`
- `spark.master`
- `spark.driver.memory`
- `spark.executor.memory`
- `spark.executor.cores`
- `spark.sql.shuffle.partitions`
- `spark.serializer`
- `spark.dynamicAllocation.enabled`
- `spark.sql.adaptive.enabled`

`SparkConf` does not process data. It provides settings used while initializing Spark.

In [ ]:
conf = (
    SparkConf()
    .setAppName("SparkLifecycleDemo")
    .setMaster("local[2]")
    .set("spark.sql.shuffle.partitions", "4")
    .set("spark.ui.showConsoleProgress", "true")
)

print("Application Name:", conf.get("spark.app.name"))
print("Master:", conf.get("spark.master"))
print("Shuffle Partitions:", conf.get("spark.sql.shuffle.partitions"))

print("\nExplicit SparkConf values:")
for key, value in conf.getAll():
    print(f"{key} = {value}")

## Important SparkConf methods

| Method | Purpose |
|---|---|
| `setAppName()` | Set application name |
| `setMaster()` | Set Spark master |
| `set()` | Set a property |
| `get()` | Read a property |
| `getAll()` | Return all explicitly set properties |
| `contains()` | Check whether a property exists |
| `setIfMissing()` | Set only when absent |

# 3. SparkContext

`SparkContext` is the lower-level connection between the application and Spark execution.

It is responsible for:

- Connecting to the execution environment
- Communicating with the cluster manager
- Creating RDDs
- Scheduling work
- Managing broadcast variables
- Managing accumulators
- Tracking application information
- Coordinating with executors

```text
SparkContext
│
├── DAGScheduler
├── TaskScheduler
├── Scheduler Backend
├── Block Manager Master
├── Broadcast Manager
├── Listener Bus
└── RDD API
```

A normal Spark application should have one active SparkContext.

# 4. SparkSession

`SparkSession` is the primary entry point for modern Spark applications.

It provides:

- DataFrame creation
- File reading and writing
- Spark SQL
- Catalog access
- Table operations
- Runtime SQL configuration
- Structured Streaming entry points
- Access to the underlying SparkContext

In [ ]:
spark = (
    SparkSession.builder
    .appName("SparkSessionContextConfLifecycle")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark Version:", spark.version)
print("Application Name:", sc.appName)
print("Application ID:", sc.applicationId)
print("Master:", sc.master)
print("Default Parallelism:", sc.defaultParallelism)
print("Spark UI:", sc.uiWebUrl)

# 5. Relationship Between SparkSession, SparkContext, and SparkConf

```text
SparkConf
   │
   ▼
SparkContext
   │
   ▼
SparkSession
   │
   ▼
DataFrame / SQL / Catalog Operations
```

| Component | Main responsibility |
|---|---|
| `SparkConf` | Stores application configuration |
| `SparkContext` | Connects the application to Spark execution |
| `SparkSession` | Provides DataFrame, SQL, catalog, and modern APIs |

In modern PySpark, we normally create a SparkSession. Spark then creates or reuses the SparkContext.

In [ ]:
print("SparkSession type:", type(spark))
print("SparkContext type:", type(sc))
print("SparkConf type:", type(sc.getConf()))
print("SparkSession uses the same SparkContext:", spark.sparkContext is sc)

# 6. SparkSession Builder Explained

```python
SparkSession.builder
```

Starts the builder.

```python
.appName("ApplicationName")
```

Sets the application name.

```python
.master("local[2]")
```

Defines the execution environment.

```python
.config("key", "value")
```

Adds a configuration.

```python
.enableHiveSupport()
```

Enables Hive integration when supported.

```python
.getOrCreate()
```

Returns an existing session or creates a new one.

# 7. Inspect Running Configuration

There are three important configuration situations:

1. Explicitly supplied properties
2. Environment or platform defaults
3. Spark built-in defaults

A property may be active even when it is not explicitly listed in `sc.getConf()`.

In [ ]:
important_properties = [
    "spark.app.name",
    "spark.master",
    "spark.driver.memory",
    "spark.executor.memory",
    "spark.executor.cores",
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
    "spark.serializer"
]

running_conf = sc.getConf()

for name in important_properties:
    print(f"{name}: {running_conf.get(name, 'Not explicitly set')}")

In [ ]:
print("All SparkContext configuration values:\n")
for key, value in sorted(running_conf.getAll()):
    print(f"{key} = {value}")

# 8. SparkContext Configuration vs Spark SQL Runtime Configuration

Use:

```python
sc.getConf()
```

to inspect application-level SparkContext configuration.

Use:

```python
spark.conf
```

for Spark SQL runtime configuration.

Many `spark.sql.*` properties can be updated during the active session.

In [ ]:
print("Current shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

spark.conf.set("spark.sql.shuffle.partitions", "6")

print("Updated shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

# 9. Static and Runtime Configurations

## Startup or static-style configurations

These normally need to be decided before SparkContext starts:

- `spark.master`
- `spark.app.name`
- `spark.driver.memory`
- `spark.executor.memory`
- `spark.executor.cores`
- `spark.serializer`

## Runtime SQL configurations

These can commonly be changed during a session:

- `spark.sql.shuffle.partitions`
- `spark.sql.adaptive.enabled`
- `spark.sql.autoBroadcastJoinThreshold`
- `spark.sql.files.maxPartitionBytes`

Not every Spark property is runtime-modifiable.

In [ ]:
print("AQE before:", spark.conf.get("spark.sql.adaptive.enabled"))

spark.conf.set("spark.sql.adaptive.enabled", "false")
print("AQE after disabling:", spark.conf.get("spark.sql.adaptive.enabled"))

spark.conf.set("spark.sql.adaptive.enabled", "true")
print("AQE after enabling:", spark.conf.get("spark.sql.adaptive.enabled"))

# 10. Configuration Precedence

A simplified precedence model is:

```text
Higher Priority
│
├── Application or builder configuration
├── spark-submit command-line options
├── --conf properties
├── spark-defaults.conf
├── Platform or environment defaults
└── Spark built-in defaults
│
Lower Priority
```

The exact behavior can depend on the property and cluster environment.

The main teaching rule is:

> A value supplied closer to application submission usually overrides a general default.

## Common configuration methods

### Application code

```python
SparkSession.builder.config(
    "spark.sql.shuffle.partitions", "100"
)
```

### spark-submit

```bash
spark-submit \
  --master yarn \
  --deploy-mode cluster \
  --driver-memory 4g \
  --executor-memory 8g \
  --executor-cores 4 \
  --num-executors 10 \
  --conf spark.sql.shuffle.partitions=200 \
  application.py
```

### spark-defaults.conf

```text
spark.executor.memory            8g
spark.executor.cores             4
spark.sql.shuffle.partitions     200
```

### Managed platforms

- Databricks cluster configuration
- AWS Glue job configuration
- Amazon EMR configuration
- Kubernetes SparkApplication specification

# 11. Why Production Code Should Avoid Hardcoding Resources

Avoid placing environment-specific resource settings inside business logic.

Better production code:

```python
spark = (
    SparkSession.builder
    .appName("DailySalesAggregation")
    .getOrCreate()
)
```

Then provide resources during submission.

Benefits:

- Same code works in development, testing, and production
- Easier CI/CD
- Easier resource tuning
- Clear separation of application logic and infrastructure

# 12. Complete Spark Application Lifecycle

```text
1. Application is submitted
        │
2. Configuration is loaded
        │
3. Driver process starts
        │
4. SparkConf is prepared
        │
5. SparkContext initializes
        │
6. Driver connects to cluster manager
        │
7. Resources are requested
        │
8. Executors are launched
        │
9. Executors register with driver
        │
10. SparkSession becomes available
        │
11. Transformations build plans
        │
12. Action triggers a job
        │
13. Job is divided into stages
        │
14. Stages are divided into tasks
        │
15. Executors process partitions
        │
16. Results are returned or written
        │
17. Application completes
        │
18. SparkContext stops
        │
19. Executors terminate
        │
20. Resources are released
```

# 13. Application Submission and Driver Startup

Applications can start through:

- Jupyter Notebook
- PySpark shell
- `spark-submit`
- Databricks Jobs
- AWS Glue
- Amazon EMR Steps
- Airflow
- Kubernetes

During driver startup:

- User code is loaded
- Configuration is loaded
- Logging initializes
- SparkContext is created
- Scheduler components start
- Spark UI starts
- Cluster-manager communication begins

# 14. Driver-Side Components

```text
Driver
│
├── SparkSession
├── SparkContext
├── DAGScheduler
├── TaskScheduler
├── Scheduler Backend
├── Block Manager Master
├── Broadcast Manager
├── Listener Bus
└── Spark UI
```

## DAGScheduler

- Converts jobs into stages
- Identifies shuffle boundaries
- Creates task sets
- Tracks stage completion

## TaskScheduler

- Receives task sets
- Assigns tasks to available executor slots
- Handles task retries
- Considers data locality

## Scheduler Backend

- Communicates with executors and cluster resources
- Tracks available executors
- Launches tasks

# 15. Cluster Manager and Executor Registration

```text
Driver
   │
   ▼
Cluster Manager
   │
   ├── Allocates CPU
   ├── Allocates Memory
   └── Starts Executors
```

Executor registration:

```text
Executor Process Starts
        │
        ▼
Executor Initializes
        │
        ▼
Executor Contacts Driver
        │
        ▼
Driver Registers Executor
        │
        ▼
Executor Becomes Available
        │
        ▼
Driver Schedules Tasks
```

Executors execute tasks; they do not create the application's logical plan.

# 16. Hands-On Lifecycle Example

Create a sales DataFrame.

In [ ]:
sales_data = [
    (1, "North", "Laptop", 50000),
    (2, "South", "Phone", 30000),
    (3, "North", "Phone", 25000),
    (4, "East", "Laptop", 60000),
    (5, "West", "Tablet", 20000),
    (6, "South", "Laptop", 55000),
    (7, "East", "Phone", 35000),
    (8, "West", "Laptop", 45000)
]

sales_columns = ["sale_id", "region", "product", "amount"]

sales_df = spark.createDataFrame(sales_data, sales_columns)

sales_df.show(truncate=False)
sales_df.printSchema()

print("Input partitions:", sales_df.rdd.getNumPartitions())

Create transformations without requesting the final result.

In [ ]:
filtered_sales_df = sales_df.filter(col("amount") >= 30000)

regional_summary_df = (
    filtered_sales_df
    .groupBy("region")
    .agg(
        count("*").alias("transaction_count"),
        spark_sum("amount").alias("total_amount"),
        avg("amount").alias("average_amount")
    )
)

print("Transformation plan created.")
print("The final grouped result has not yet been requested.")

# 17. Logical and Physical Plans

Spark creates:

```text
Parsed Logical Plan
        │
Analyzed Logical Plan
        │
Optimized Logical Plan
        │
Physical Plan
        │
Tasks
```

The optimized plan may apply:

- Predicate pushdown
- Column pruning
- Constant folding
- Null propagation
- Join selection
- Aggregate optimization

In [ ]:
regional_summary_df.explain(mode="extended")

# 18. Action, Job, Stage, and Task Creation

The following action triggers execution.

In [ ]:
regional_summary_df.show(truncate=False)

Execution flow:

```text
Action
  │
  ▼
Job
  │
  ├── Stage 1
  │     ├── Task 1
  │     ├── Task 2
  │     └── Task 3
  │
  └── Stage 2
        ├── Task 1
        ├── Task 2
        └── Task 3
```

`groupBy()` normally creates a shuffle boundary.

```text
Stage 1
├── Read
├── Filter
└── Shuffle Write
       │
       ▼
     Shuffle
       │
       ▼
Stage 2
├── Shuffle Read
└── Aggregate
```

In [ ]:
print("Input partitions:", sales_df.rdd.getNumPartitions())
print("Summary partitions:", regional_summary_df.rdd.getNumPartitions())
print("Configured shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

With Adaptive Query Execution enabled, Spark may coalesce shuffle partitions at runtime. Therefore, the observed partition count may differ from the configured default.

# 19. Task Scheduling and Executor Execution

For a stage:

```text
Partition 0 → Task 0
Partition 1 → Task 1
Partition 2 → Task 2
Partition 3 → Task 3
```

If two cores are available:

```text
Wave 1: Task 0 and Task 1
Wave 2: Task 2 and Task 3
```

An executor task:

1. Receives serialized instructions
2. Reads one assigned partition
3. Executes transformations
4. Produces intermediate or final output
5. Writes shuffle data when necessary
6. Reports completion or failure

# 20. Task, Executor, and Driver Failures

## Task failure

Spark normally retries a failed task up to configured limits.

## Executor failure

- Running tasks are lost
- Cached blocks may be lost
- Tasks can be rescheduled
- Replacement resources may be created

## Driver failure

- Scheduling state is lost
- Executors cannot receive new instructions
- The application normally terminates

Task retry is Spark-level fault tolerance. Driver restart is usually handled by the external platform or orchestration system.

# 21. Broadcast Variables and Accumulators

These are accessed through SparkContext.

## Broadcast variable

Efficiently distributes read-only data.

## Accumulator

Allows executors to add to a driver-visible counter.

Accumulators should not be used as transactional business logic because task retries can repeat updates.

In [ ]:
tax_rates = {
    "North": 0.10,
    "South": 0.12,
    "East": 0.08,
    "West": 0.09
}

broadcast_tax_rates = sc.broadcast(tax_rates)
print("Broadcast value:", broadcast_tax_rates.value)

record_counter = sc.accumulator(0)

def count_record(value):
    record_counter.add(1)
    return value

_ = sc.parallelize(range(10), 2).map(count_record).collect()
print("Accumulator value:", record_counter.value)

# 22. SparkSession Catalog and SQL

In [ ]:
sales_df.createOrReplaceTempView("sales")

query = '''
SELECT
    region,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_amount
FROM sales
GROUP BY region
ORDER BY region
'''

spark.sql(query).show()

A temporary view belongs to the SparkSession and normally disappears when the session ends.

# 23. Multiple SparkSessions and One SparkContext

A new SparkSession can share the same SparkContext.

In [ ]:
second_session = spark.newSession()

print("Different SparkSession object:", second_session is not spark)
print("Same SparkContext:", second_session.sparkContext is spark.sparkContext)

print(
    "Original session shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

second_session.conf.set("spark.sql.shuffle.partitions", "3")

print(
    "Second session shuffle partitions:",
    second_session.conf.get("spark.sql.shuffle.partitions")
)

print(
    "Original session shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

Multiple sessions can have separate SQL configuration and temporary views while sharing the same SparkContext and Spark application.

# 24. Understanding `getOrCreate()`

Calling the builder again normally reuses an existing session or context.

In [ ]:
another_reference = (
    SparkSession.builder
    .appName("AnotherRequestedName")
    .getOrCreate()
)

print("Same SparkSession object:", another_reference is spark)
print("Running application name:", another_reference.sparkContext.appName)

## Common notebook issue

Suppose Spark already started with:

```python
.master("local[2]")
```

Later you request:

```python
.master("local[8]")
```

`getOrCreate()` may reuse the original context, so the master remains `local[2]`.

For startup-level changes:

1. Stop the existing application
2. Restart the notebook kernel
3. Create a fresh SparkSession

Restarting the kernel is usually the safest approach.

# 25. One SparkContext Rule

A normal process should have one active SparkContext.

Multiple contexts can cause:

- Port conflicts
- Resource contention
- Scheduler confusion
- Unexpected errors
- Difficult debugging

Create SparkSession and reuse its SparkContext.

# 26. Application Shutdown

A Spark application stops when:

- `spark.stop()` is called
- `sc.stop()` is called
- The driver exits
- The notebook kernel stops
- The cluster manager kills the application
- An unrecoverable failure occurs

Shutdown flow:

```text
spark.stop()
    │
    ▼
SparkSession Stops
    │
    ▼
SparkContext Stops
    │
    ├── Schedulers Stop
    ├── Spark UI Stops
    ├── Executors Terminate
    ├── Network Services Stop
    └── Resources Are Released
```

# 27. Graceful and Forced Shutdown

## Graceful shutdown

- Running work completes
- Output is finalized
- Logs are finalized
- Executors are released cleanly

## Forced termination

- Tasks are cancelled
- Temporary or partial output may remain
- Cleanup may be required
- Logs may show abrupt failure

Production writes should be idempotent and recoverable.

# 28. Spark UI and History Server

The live Spark UI is hosted by the driver.

```text
http://localhost:4040
```

After the application stops, the live UI normally disappears.

Event logging can preserve execution information:

```text
spark.eventLog.enabled true
spark.eventLog.dir     file:///path/to/event-logs
```

Spark History Server reads these event logs and displays completed applications.

In [ ]:
print("Current Spark UI URL:", sc.uiWebUrl)

# 29. Local, Client, and Cluster Lifecycle

## Local mode

```text
Notebook / Python Process
        │
        ▼
Driver JVM
        │
        ▼
Local Scheduler
        │
        ▼
Local Task Threads
```

## Client mode

```text
Submission Machine
├── Driver
└── Spark UI
      │
      ▼
Cluster Executors
```

## Cluster mode

```text
Submission Client
      │
      ▼
Cluster Manager
      │
      ▼
Driver in Cluster
      │
      ▼
Executors in Cluster
```

Cluster mode is commonly preferred for scheduled production workloads.

# 30. Managed Platform Examples

## Databricks

- Driver and workers run on managed compute
- Notebook receives a SparkSession
- Spark UI is integrated into the platform

## AWS Glue

- Managed Spark workers are provisioned
- Driver and executors run in serverless infrastructure
- Logs are written to CloudWatch

## Amazon EMR

- Spark often runs through YARN
- Driver and executor containers are allocated
- Logs can be stored on the cluster and in S3

# 31. Common Configuration Mistakes

- Setting executor properties in local mode and expecting distributed executors
- Changing driver memory after SparkContext starts
- Assuming more driver memory speeds up executor processing
- Using too many cores per executor
- Using too few shuffle partitions
- Using too many tiny shuffle partitions
- Hardcoding infrastructure settings in business code
- Confusing `sc.getConf()` with `spark.conf`
- Creating multiple SparkContexts
- Forgetting that `getOrCreate()` may reuse an existing context

# 32. Spark UI Exercise

Create a transformation chain.

In [ ]:
lifecycle_df = (
    sales_df
    .filter(col("amount") > 25000)
    .groupBy("product")
    .agg(
        count("*").alias("sales_count"),
        spark_sum("amount").alias("total_sales")
    )
    .orderBy(col("total_sales").desc())
)

print("Transformation chain created.")

In [ ]:
lifecycle_df.explain(mode="formatted")

In [ ]:
lifecycle_df.show(truncate=False)

Open the Spark UI and inspect:

- Jobs
- Stages
- SQL
- Executors
- Environment

Find:

1. The action that triggered execution
2. Number of jobs
3. Number of stages
4. Shuffle write
5. Shuffle read
6. Number of tasks
7. `Exchange`
8. Aggregate operators
9. Current application properties

# 33. Interview Questions and Answers

## What is SparkSession?

SparkSession is the modern high-level entry point for DataFrame, SQL, catalog, table, and streaming operations.

## What is SparkContext?

SparkContext is the low-level connection between a Spark application and its execution environment.

## What is SparkConf?

SparkConf stores application-level Spark configuration.

## How are they related?

SparkConf provides configuration, SparkContext initializes execution using that configuration, and SparkSession provides higher-level APIs over SparkContext.

## Can multiple SparkSessions exist?

Yes. Multiple SparkSessions can share one SparkContext.

## Can multiple SparkContexts exist?

A normal process should use one active SparkContext.

## What does `getOrCreate()` do?

It returns an existing session when available or creates a new one.

## Can Spark master change after startup?

Not for the running SparkContext. It is a startup-level property.

## Can shuffle partitions change at runtime?

Yes, commonly through `spark.conf.set()`.

## What happens when an action is called?

Spark creates a job, DAGScheduler creates stages, TaskScheduler assigns tasks, and executors process partitions.

## What happens during `spark.stop()`?

Schedulers stop, executors terminate, UI services stop, and application resources are released.

# 34. Scenario-Based Questions

## Configuration does not change in a notebook

An existing SparkContext is being reused. Restart the kernel for startup-level changes.

## Executor memory changed in code but not in cluster

Executor memory is decided before executors launch. Supply it during submission or cluster configuration.

## Spark UI disappeared after completion

The live UI belongs to the driver. Use event logs and History Server for completed applications.

## One task failed but the application continued

Spark retried the failed task.

## An executor failed and cached data disappeared

The cached blocks were stored on that executor. Spark may recompute them using lineage.

# 35. Knowledge Check

1. What is the purpose of SparkConf?
2. What is the purpose of SparkContext?
3. What is the purpose of SparkSession?
4. Can sessions share a context?
5. Why should one process normally have one context?
6. Which component creates stages?
7. Which component assigns tasks?
8. When do executors register?
9. What triggers a job?
10. Which properties should be set before startup?
11. Which SQL properties can change at runtime?
12. Why may notebook builder changes not apply?
13. What happens when an executor fails?
14. What happens when the driver fails?
15. How can completed applications be inspected?

# 36. Key Points to Remember

```text
1. SparkConf stores application configuration.
2. SparkContext connects the application to execution.
3. SparkSession is the modern high-level entry point.
4. SparkSession provides access to SparkContext.
5. A normal application has one active SparkContext.
6. Multiple SparkSessions can share one SparkContext.
7. getOrCreate() may reuse an existing context.
8. Startup properties must be set before context creation.
9. Many spark.sql properties can change at runtime.
10. DAGScheduler creates stages.
11. TaskScheduler assigns tasks.
12. Executors register before receiving tasks.
13. Actions create jobs.
14. Tasks process partitions.
15. Driver failure usually ends the application.
16. Executor failure can often be recovered.
17. Spark UI is hosted by the driver.
18. History Server displays event-log history.
19. spark.stop() releases application resources.
20. Production code should separate logic from infrastructure configuration.
```

# 37. Optional Cleanup

Run this only after all Spark UI observations are complete.

In [ ]:
# Uncomment only after completing the notebook.
# spark.stop()